In [ ]:
# Configure Kaggle credentials outside this notebook.
# Use ~/.kaggle/access_token locally or Kaggle Secrets in cloud notebooks.


# Notebook 1 (Full Run): 3-Epoch QLoRA Fine-Tuning for Text-to-SQL
### Target: `unsloth/Qwen2.5-Coder-7B-Instruct` on Spider Dataset (3 Full Epochs)

This notebook executes the full **3-epoch** training cycle across the Spider training split (7,000 samples $\times$ 3 = 21,000 example presentations).

**Training parameters:**
- **Epochs**: 3 full epochs (2,625 total optimization steps)
- **Optimizer**: `adamw_8bit` with Cosine learning rate scheduling
- **Target Modules**: `q, k, v, o, gate, up, down` (rank $r=16, \alpha=16$)
- **Batch configuration**: Batch size 2, gradient accumulation 4 (effective batch size = 8)
- **Checkpointing**: Checkpoint saved after each epoch

## 1. Environment & Dependency Installation

In [ ]:
%%capture
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir trl peft bitsandbytes datasets transformers huggingface_hub accelerate

## 2. Hardware Diagnostics

In [ ]:
import os
import torch

assert torch.cuda.is_available(), "CUDA GPU not detected! Switch accelerator to T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"[Hardware Init] GPU: {gpu_name} | VRAM: {gpu_memory_gb:.2f} GB")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 3. Base Model & QLoRA Configuration

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
BASE_MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct"
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("[Model Ready] Trainable parameters summary:")
model.print_trainable_parameters()

## 4. Dataset Ingestion & Chat Template Formatting

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("xlangai/spider", split="train")
print(f"[Dataset] Ingested {len(raw_dataset)} training records from 'xlangai/spider'.")

SYSTEM_PROMPT = "You are an expert SQL assistant. Given a database schema and a question, write a valid SQL query that answers the question."

def format_spider_prompts(batch):
    formatted_texts = []
    for db_id, question, query in zip(batch["db_id"], batch["question"], batch["query"]):
        convo = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Database: {db_id}\nQuestion: {question}"},
            {"role": "assistant", "content": f"```sql\n{query}\n```"},
        ]
        formatted_text = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted_texts.append(formatted_text)
    return {"text": formatted_texts}

train_dataset = raw_dataset.map(
    format_spider_prompts,
    batched=True,
    remove_columns=raw_dataset.column_names,
    desc="Formatting Chat Templates",
)
print("[Dataset Ready] Sample conversation formatted:")
print(train_dataset[0]["text"])

## 5. Full 3-Epoch SFT Training Pipeline
Training for 3 complete epochs (875 steps per epoch $\times 3 = 2,625$ steps).

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir="outputs_qlora_sql_3epochs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=3,                  # Full 3 Epochs
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=25,                    # Periodic logging
    save_strategy="epoch",               # Checkpoint at end of each epoch
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",          # Cosine decay for smooth multi-epoch convergence
    seed=3407,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("[Training] Launching 3-epoch SFT optimization loop...")
train_stats = trainer.train()
print(f"[Training Complete] Peak memory: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")

## 6. Inference checks: aggregate, JOIN, and GROUP BY

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    {
        "name": "Test 1 (Simple Aggregate)",
        "prompt": "Database: concert_singer\nQuestion: How many singers do we have?",
    },
    {
        "name": "Test 2 (JOIN Query)",
        "prompt": "Database: concert_singer\nQuestion: What are the names of all singers who had a concert in 2020?",
    },
    {
        "name": "Test 3 (GROUP BY + HAVING)",
        "prompt": "Database: concert_singer\nQuestion: Show the country and number of singers for countries with more than 1 singer.",
    },
]

for test in test_cases:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test["prompt"]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        use_cache=True,
        temperature=0.1,
    )
    pred = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n=== {test['name']} ===")
    print(pred)

## 7. LoRA Adapter Export

In [ ]:
import huggingface_hub

HF_TOKEN = "hf_YOUR_TOKEN_HERE"
HF_REPO_NAME = "username/qwen2.5-coder-7b-text2sql-3epochs"
LOCAL_SAVE_DIR = "lora_sql_adapter_3epochs"

# 1. Save local LoRA weights (~50MB)
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)
print(f"[Export] Saved 3-epoch LoRA adapter weights to ./{LOCAL_SAVE_DIR}/")

# 2. Authenticate and push to Hub if token provided
if HF_TOKEN.startswith("hf_") and HF_TOKEN != "hf_YOUR_TOKEN_HERE":
    huggingface_hub.login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    print("[Export Complete] Published to Hugging Face Hub.")
else:
    print("[Notice] Local weights ready in ./{LOCAL_SAVE_DIR}/")